<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Python profesional para analítica avanzada</p>
  </div>

</div>

# **Iteradores, generadores y procesamiento eficiente de ficheros grandes**

## Python profesional para analítica avanzada

Este laboratorio estudia cómo procesar datos cuando **no queremos o no podemos cargarlo todo en memoria**.

La idea central será:

> **si el problema puede resolverse elemento a elemento o bloque a bloque, no siempre necesitamos materializar todos los datos simultáneamente.**

Trabajaremos con:

- iterables;
- iteradores;
- `iter()` y `next()`;
- generadores;
- `yield`;
- generator expressions;
- memoria de listas vs generadores;
- lectura línea a línea;
- CSV por chunks;
- acumuladores;
- agregaciones incrementales;
- errores típicos al procesar por bloques;
- validación contra procesamiento completo;
- Polars lazy y `scan_csv`;
- comparación de memoria, tiempo y escalabilidad.

## Objetivos

Al finalizar deberías poder:

1. Explicar la diferencia entre iterable e iterador.
2. Utilizar `iter()` y `next()`.
3. Comprender el agotamiento de un iterador.
4. Crear generadores con `yield`.
5. Utilizar generator expressions.
6. Comparar memoria de listas y generadores.
7. Leer ficheros línea a línea.
8. Procesar CSV por chunks.
9. Diseñar acumuladores incrementales.
10. Calcular medias globales correctamente por bloques.
11. Evitar la “media de medias” incorrecta.
12. Combinar estadísticas parciales.
13. Procesar ficheros mayores que la memoria disponible.
14. Validar resultados contra una carga completa.
15. Utilizar Polars `scan_csv`.
16. Entender por qué lazy execution puede evitar materialización innecesaria.
17. Comparar enfoques en tiempo y memoria.

## 1. Preparación

In [ ]:
from pathlib import Path
import sys
import time
import tracemalloc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)

try:
    import polars as pl
    POLARS_DISPONIBLE = True
    print("Polars:", pl.__version__)
except ImportError:
    POLARS_DISPONIBLE = False
    print("Polars no instalado. Ejecuta: pip install polars")

print("Pandas:", pd.__version__)

# 2. Iterable vs iterador

In [ ]:
valores = [10, 20, 30, 40]

print(
    "Iterable:",
    valores
)

iterador = iter(valores)

print(
    "Tipo del iterador:",
    type(iterador)
)

### Conceptos

Un **iterable** es un objeto sobre el que podemos iterar.

Un **iterador** mantiene estado y entrega el siguiente elemento cuando llamamos a `next()`.

# 3. `next()`

In [ ]:
print(next(iterador))
print(next(iterador))
print(next(iterador))
print(next(iterador))

# 4. Provocar agotamiento

In [ ]:
try:
    print(
        next(iterador)
    )
except StopIteration as error:
    print(
        type(error).__name__,
        "-> el iterador se ha agotado"
    )

### Interpretación

Los iteradores se consumen.

Una vez agotados, no se “reinician” automáticamente.

# 5. Crear un nuevo iterador

In [ ]:
iterador_nuevo = iter(
    valores
)

print(
    list(iterador_nuevo)
)

# 6. List comprehension

In [ ]:
cuadrados_lista = [
    x ** 2
    for x in range(10)
]

cuadrados_lista

# 7. Generator expression

In [ ]:
cuadrados_generador = (
    x ** 2
    for x in range(10)
)

cuadrados_generador

## 7.1 Consumir generador

In [ ]:
for valor in cuadrados_generador:
    print(valor)

El generador no calcula todos los valores al crearse. Los produce bajo demanda.

# 8. Memoria: lista vs generador

In [ ]:
n = 1_000_000

lista_grande = [
    i ** 2
    for i in range(n)
]

generador_grande = (
    i ** 2
    for i in range(n)
)

print(
    "Lista:",
    sys.getsizeof(
        lista_grande
    ),
    "bytes"
)

print(
    "Generador:",
    sys.getsizeof(
        generador_grande
    ),
    "bytes"
)

### Interpretación

`sys.getsizeof()` mide el objeto contenedor, no toda la memoria profunda de todos los enteros.

Aun así, ilustra la diferencia fundamental:

- la lista materializa referencias a todos los resultados;
- el generador mantiene únicamente el estado necesario para producir el siguiente.

# 9. Visualizar memoria aproximada

In [ ]:
comparacion_mem = pd.DataFrame({
    "estructura": [
        "Lista",
        "Generador"
    ],
    "bytes_objeto": [
        sys.getsizeof(
            lista_grande
        ),
        sys.getsizeof(
            generador_grande
        )
    ]
})

fig, ax = plt.subplots(figsize=(6, 4))

ax.bar(
    comparacion_mem["estructura"],
    comparacion_mem["bytes_objeto"]
)

ax.set(
    title="Tamaño del objeto contenedor",
    ylabel="Bytes"
)

plt.show()

# 10. Crear generadores con `yield`

In [ ]:
def generar_cuadrados(
    n
):
    for i in range(n):
        yield i ** 2


generador = generar_cuadrados(
    5
)

print(
    next(generador)
)

print(
    next(generador)
)

### `yield`

`yield` pausa la función y conserva su estado.

Cuando se solicita el siguiente valor, continúa desde el punto donde se detuvo.

# 11. Inspeccionar comportamiento perezoso

In [ ]:
def generador_verbose(
    n
):
    for i in range(n):
        print(
            "Generando:",
            i
        )
        yield i


g = generador_verbose(
    3
)

print(
    "Generador creado"
)

print(
    "Primer next:"
)

print(
    next(g)
)

### Interpretación

Crear el generador no ejecuta inmediatamente todo su cuerpo.

La ejecución ocurre conforme consumimos valores.

# 12. Función normal vs generador

In [ ]:
def construir_lista(
    n
):
    resultado = []

    for i in range(n):
        resultado.append(
            i ** 2
        )

    return resultado


def producir_valores(
    n
):
    for i in range(n):
        yield i ** 2

# 13. Medir pico de memoria

In [ ]:
def medir_memoria(
    funcion
):
    tracemalloc.start()

    inicio = time.perf_counter()
    resultado = funcion()
    duracion = (
        time.perf_counter()
        - inicio
    )

    actual, pico = (
        tracemalloc
        .get_traced_memory()
    )

    tracemalloc.stop()

    return {
        "tiempo_ms":
            duracion * 1000,
        "pico_MB":
            pico / 1024**2,
        "resultado":
            resultado
    }


mem_lista = medir_memoria(
    lambda:
        construir_lista(
            500_000
        )
)

mem_generador = medir_memoria(
    lambda:
        producir_valores(
            500_000
        )
)

pd.DataFrame({
    "metodo": [
        "Lista materializada",
        "Generador creado"
    ],
    "tiempo_ms": [
        mem_lista["tiempo_ms"],
        mem_generador["tiempo_ms"]
    ],
    "pico_MB": [
        mem_lista["pico_MB"],
        mem_generador["pico_MB"]
    ]
})

### Cuidado al interpretar

Aquí estamos comparando:

- construir todos los valores;
- crear un generador sin consumirlo.

Eso responde a una pregunta concreta: **coste de materialización inicial**.

No sería una comparación justa para medir el coste total de procesar todos los elementos.

# 14. Benchmark justo: consumir todo

In [ ]:
def consumir_generador(
    n
):
    total = 0

    for valor in producir_valores(
        n
    ):
        total += valor

    return total


def consumir_lista(
    n
):
    valores = construir_lista(
        n
    )

    total = 0

    for valor in valores:
        total += valor

    return total


mem_consumir_lista = medir_memoria(
    lambda:
        consumir_lista(
            500_000
        )
)

mem_consumir_gen = medir_memoria(
    lambda:
        consumir_generador(
            500_000
        )
)

pd.DataFrame({
    "metodo": [
        "Lista + consumo",
        "Generador + consumo"
    ],
    "tiempo_ms": [
        mem_consumir_lista["tiempo_ms"],
        mem_consumir_gen["tiempo_ms"]
    ],
    "pico_MB": [
        mem_consumir_lista["pico_MB"],
        mem_consumir_gen["pico_MB"]
    ]
})

# 15. Visualizar pico de memoria

In [ ]:
comparacion_consumo = pd.DataFrame({
    "metodo": [
        "Lista",
        "Generador"
    ],
    "pico_MB": [
        mem_consumir_lista["pico_MB"],
        mem_consumir_gen["pico_MB"]
    ]
})

fig, ax = plt.subplots(figsize=(6, 4))

ax.bar(
    comparacion_consumo["metodo"],
    comparacion_consumo["pico_MB"]
)

ax.set(
    title="Pico de memoria al procesar todos los elementos",
    ylabel="MB"
)

plt.show()

### Interpretación

El generador puede reducir el pico de memoria porque procesa valores de forma incremental.

No significa necesariamente que siempre sea más rápido.

# 16. Cuándo NO usar generador

Un generador puede ser incómodo cuando:

- necesitamos múltiples recorridos;
- necesitamos acceso aleatorio;
- necesitamos `len()` directamente;
- necesitamos ordenar todo el conjunto;
- el dataset cabe cómodamente en memoria y la simplicidad importa más.

La eficiencia debe evaluarse en contexto.

# 17. Crear un fichero grande de prueba

In [ ]:
N = 1_000_000

ruta_csv = Path(
    "ventas_grandes_demo.csv"
)

datos_grandes = pd.DataFrame({
    "cliente_id":
        rng.integers(
            1,
            100_001,
            N
        ),

    "region":
        rng.choice(
            [
                "Norte",
                "Sur",
                "Este",
                "Oeste",
                "Centro"
            ],
            N
        ),

    "canal":
        rng.choice(
            [
                "web",
                "tienda",
                "partner"
            ],
            N
        ),

    "unidades":
        rng.integers(
            1,
            10,
            N
        ),

    "precio":
        rng.uniform(
            5,
            300,
            N
        )
})

datos_grandes[
    "importe"
] = (
    datos_grandes["unidades"]
    * datos_grandes["precio"]
)

datos_grandes.to_csv(
    ruta_csv,
    index=False
)

print(
    "Fichero:",
    ruta_csv.resolve()
)

print(
    "Tamaño:",
    round(
        ruta_csv.stat().st_size
        / 1024**2,
        2
    ),
    "MB"
)

# 18. Problema inicial: cargar todo

In [ ]:
mem_carga_completa = medir_memoria(
    lambda:
        pd.read_csv(
            ruta_csv
        )
)

print(
    "Tiempo:",
    round(
        mem_carga_completa[
            "tiempo_ms"
        ],
        2
    ),
    "ms"
)

print(
    "Pico Python:",
    round(
        mem_carga_completa[
            "pico_MB"
        ],
        2
    ),
    "MB"
)

### Observación

La carga completa es simple, pero materializa todo el CSV en memoria.

¿Qué ocurre si solo necesitamos una agregación?

# 19. Agregación completa como baseline

In [ ]:
def resumen_completo():
    datos = pd.read_csv(
        ruta_csv
    )

    return (
        datos.groupby(
            "region",
            as_index=False
        )
        .agg(
            facturacion=(
                "importe",
                "sum"
            ),
            operaciones=(
                "importe",
                "size"
            ),
            importe_medio=(
                "importe",
                "mean"
            )
        )
        .sort_values(
            "region"
        )
        .reset_index(
            drop=True
        )
    )


baseline = medir_memoria(
    resumen_completo
)

baseline[
    "resultado"
]

# 20. `read_csv(chunksize=...)`

In [ ]:
lector_chunks = pd.read_csv(
    ruta_csv,
    chunksize=100_000
)

print(
    type(
        lector_chunks
    )
)

`TextFileReader` actúa como un iterador de DataFrames.

Cada iteración entrega un bloque.

# 21. Ver los primeros chunks

In [ ]:
lector_demo = pd.read_csv(
    ruta_csv,
    chunksize=100_000
)

for i, chunk in enumerate(
    lector_demo
):
    print(
        "Chunk:",
        i,
        "shape:",
        chunk.shape
    )

    if i == 2:
        break

# 22. Error analítico: media de medias

Supongamos que calculamos la media de `importe` por chunk y luego hacemos la media de esas medias.

Si todos los chunks tienen exactamente el mismo tamaño, puede coincidir.

Pero si el último chunk tiene distinto tamaño, deja de ser correcto.

In [ ]:
chunk_means = []
chunk_sizes = []

for chunk in pd.read_csv(
    ruta_csv,
    chunksize=130_000
):
    chunk_means.append(
        chunk["importe"].mean()
    )

    chunk_sizes.append(
        len(chunk)
    )

media_de_medias = np.mean(
    chunk_means
)

media_correcta = pd.read_csv(
    ruta_csv,
    usecols=["importe"]
)["importe"].mean()

print(
    "Media de medias:",
    media_de_medias
)

print(
    "Media correcta:",
    media_correcta
)

print(
    "Diferencia:",
    media_de_medias
    - media_correcta
)

### Diagnóstico

Los chunks no pesan lo mismo.

La media global correcta debe ponderar por número de observaciones.

# 23. Media incremental correcta

In [ ]:
suma_total = 0.0
conteo_total = 0

for chunk in pd.read_csv(
    ruta_csv,
    chunksize=130_000,
    usecols=["importe"]
):
    suma_total += (
        chunk["importe"]
        .sum()
    )

    conteo_total += (
        chunk["importe"]
        .count()
    )

media_incremental = (
    suma_total
    / conteo_total
)

print(
    "Media incremental:",
    media_incremental
)

print(
    "Media completa:",
    media_correcta
)

print(
    "Equivalentes:",
    np.isclose(
        media_incremental,
        media_correcta
    )
)

# 24. Agregación por grupos mediante chunks

In [ ]:
def resumen_chunks(
    chunksize=100_000
):
    acumulado = {}

    for chunk in pd.read_csv(
        ruta_csv,
        chunksize=chunksize
    ):
        parcial = (
            chunk.groupby(
                "region"
            )
            .agg(
                suma=(
                    "importe",
                    "sum"
                ),
                conteo=(
                    "importe",
                    "size"
                )
            )
        )

        for region, fila in parcial.iterrows():

            if region not in acumulado:
                acumulado[
                    region
                ] = {
                    "suma": 0.0,
                    "conteo": 0
                }

            acumulado[
                region
            ]["suma"] += (
                fila["suma"]
            )

            acumulado[
                region
            ]["conteo"] += int(
                fila["conteo"]
            )

    filas = []

    for region, valores in acumulado.items():
        filas.append({
            "region":
                region,

            "facturacion":
                valores["suma"],

            "operaciones":
                valores["conteo"],

            "importe_medio":
                (
                    valores["suma"]
                    / valores["conteo"]
                )
        })

    return (
        pd.DataFrame(
            filas
        )
        .sort_values(
            "region"
        )
        .reset_index(
            drop=True
        )
    )


resultado_chunks = resumen_chunks(
    chunksize=100_000
)

resultado_chunks

# 25. Validar chunks contra carga completa

In [ ]:
resultado_completo = (
    baseline[
        "resultado"
    ]
)

print(
    "Facturación:",
    np.allclose(
        resultado_completo[
            "facturacion"
        ],
        resultado_chunks[
            "facturacion"
        ]
    )
)

print(
    "Operaciones:",
    (
        resultado_completo[
            "operaciones"
        ]
        ==
        resultado_chunks[
            "operaciones"
        ]
    ).all()
)

print(
    "Media:",
    np.allclose(
        resultado_completo[
            "importe_medio"
        ],
        resultado_chunks[
            "importe_medio"
        ]
    )
)

### Regla fundamental

El procesamiento incremental solo es correcto si podemos combinar de forma válida los estadísticos parciales.

# 26. Medir memoria de chunks

In [ ]:
mem_chunks = medir_memoria(
    lambda:
        resumen_chunks(
            chunksize=100_000
        )
)

pd.DataFrame({
    "metodo": [
        "Carga completa",
        "Chunks"
    ],
    "tiempo_ms": [
        baseline[
            "tiempo_ms"
        ],
        mem_chunks[
            "tiempo_ms"
        ]
    ],
    "pico_MB": [
        baseline[
            "pico_MB"
        ],
        mem_chunks[
            "pico_MB"
        ]
    ]
})

# 27. Visualizar pico de memoria

In [ ]:
comparacion_chunks = pd.DataFrame({
    "metodo": [
        "Completo",
        "Chunks"
    ],
    "pico_MB": [
        baseline["pico_MB"],
        mem_chunks["pico_MB"]
    ]
})

fig, ax = plt.subplots(figsize=(6, 4))

ax.bar(
    comparacion_chunks[
        "metodo"
    ],
    comparacion_chunks[
        "pico_MB"
    ]
)

ax.set(
    title="Pico de memoria: carga completa vs chunks",
    ylabel="MB"
)

plt.show()

### Interpretación

El objetivo principal de chunks no es necesariamente reducir tiempo.

Es reducir el volumen de datos simultáneamente materializado.

# 28. Efecto del tamaño de chunk

In [ ]:
chunk_sizes = [
    25_000,
    50_000,
    100_000,
    250_000
]

resultados_chunk_size = []

for tamano in chunk_sizes:
    medicion = medir_memoria(
        lambda t=tamano:
            resumen_chunks(
                chunksize=t
            )
    )

    resultados_chunk_size.append({
        "chunksize":
            tamano,

        "tiempo_ms":
            medicion[
                "tiempo_ms"
            ],

        "pico_MB":
            medicion[
                "pico_MB"
            ]
    })

df_chunk_sizes = pd.DataFrame(
    resultados_chunk_size
)

df_chunk_sizes

# 29. Visualizar trade-off

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(
    df_chunk_sizes[
        "chunksize"
    ],
    df_chunk_sizes[
        "pico_MB"
    ],
    marker="o"
)

ax.set(
    title="Tamaño de chunk vs pico de memoria",
    xlabel="Chunk size",
    ylabel="Pico de memoria (MB)"
)

plt.show()

### Interpretación

Chunks más pequeños tienden a reducir memoria, pero pueden aumentar overhead de iteración y parsing.

No existe un tamaño universalmente óptimo.

# 30. Leer solo columnas necesarias

In [ ]:
def resumen_chunks_reducido(
    chunksize=100_000
):
    acumulado = {}

    for chunk in pd.read_csv(
        ruta_csv,
        usecols=[
            "region",
            "importe"
        ],
        chunksize=chunksize
    ):
        parcial = (
            chunk.groupby(
                "region"
            )["importe"]
            .agg(
                ["sum", "count"]
            )
        )

        for region, fila in parcial.iterrows():

            if region not in acumulado:
                acumulado[
                    region
                ] = [
                    0.0,
                    0
                ]

            acumulado[
                region
            ][0] += (
                fila["sum"]
            )

            acumulado[
                region
            ][1] += int(
                fila["count"]
            )

    return acumulado


medicion_reducida = medir_memoria(
    resumen_chunks_reducido
)

print(
    "Pico MB:",
    medicion_reducida[
        "pico_MB"
    ]
)

Seleccionar columnas en lectura evita materializar variables que no participan en el cálculo.

# 31. Procesamiento línea a línea

In [ ]:
ruta_txt = Path(
    "numeros_demo.txt"
)

with ruta_txt.open(
    "w",
    encoding="utf-8"
) as fichero:

    for i in range(
        100_000
    ):
        fichero.write(
            f"{i}\n"
        )

print(
    ruta_txt.stat().st_size
    / 1024**2,
    "MB"
)

# 32. Mala práctica: `readlines()`

In [ ]:
def suma_readlines():
    with ruta_txt.open(
        "r",
        encoding="utf-8"
    ) as fichero:

        lineas = (
            fichero.readlines()
        )

    return sum(
        int(
            linea.strip()
        )
        for linea in lineas
    )

# 33. Lectura incremental

In [ ]:
def suma_linea_a_linea():
    total = 0

    with ruta_txt.open(
        "r",
        encoding="utf-8"
    ) as fichero:

        for linea in fichero:
            total += int(
                linea.strip()
            )

    return total

# 34. Comparar memoria

In [ ]:
mem_readlines = medir_memoria(
    suma_readlines
)

mem_stream = medir_memoria(
    suma_linea_a_linea
)

pd.DataFrame({
    "metodo": [
        "readlines",
        "iteración fichero"
    ],
    "pico_MB": [
        mem_readlines[
            "pico_MB"
        ],
        mem_stream[
            "pico_MB"
        ]
    ],
    "tiempo_ms": [
        mem_readlines[
            "tiempo_ms"
        ],
        mem_stream[
            "tiempo_ms"
        ]
    ]
})

# 35. Validar equivalencia

In [ ]:
print(
    suma_readlines()
    ==
    suma_linea_a_linea()
)

# 36. Generador para lectura de registros

In [ ]:
def leer_numeros(
    ruta
):
    with Path(
        ruta
    ).open(
        "r",
        encoding="utf-8"
    ) as fichero:

        for linea in fichero:
            yield int(
                linea.strip()
            )


g = leer_numeros(
    ruta_txt
)

print(
    next(g)
)

print(
    next(g)
)

# 37. Pipeline de generadores

In [ ]:
def filtrar_pares(
    iterable
):
    for valor in iterable:
        if (
            valor % 2 == 0
        ):
            yield valor


def elevar_cuadrado(
    iterable
):
    for valor in iterable:
        yield (
            valor ** 2
        )


pipeline = elevar_cuadrado(
    filtrar_pares(
        leer_numeros(
            ruta_txt
        )
    )
)

primeros = []

for i, valor in enumerate(
    pipeline
):
    primeros.append(
        valor
    )

    if i == 9:
        break

primeros

### Interpretación

Cada etapa procesa valores bajo demanda.

No necesitamos crear una lista intermedia de números pares ni otra lista de cuadrados.

# 38. Preparar Polars lazy

In [ ]:
if POLARS_DISPONIBLE:
    lazy_csv = (
        pl.scan_csv(
            ruta_csv
        )
    )

    print(
        lazy_csv
    )
else:
    print(
        "Instalar Polars para continuar."
    )

`scan_csv()` no carga inmediatamente todo el fichero.

Construye un `LazyFrame`.

# 39. Consulta lazy

In [ ]:
if POLARS_DISPONIBLE:
    consulta_lazy = (
        pl.scan_csv(
            ruta_csv
        )
        .select(
            "region",
            "importe"
        )
        .filter(
            pl.col(
                "importe"
            )
            > 500
        )
        .group_by(
            "region"
        )
        .agg(
            pl.col(
                "importe"
            )
            .sum()
            .alias(
                "facturacion"
            ),

            pl.len()
            .alias(
                "operaciones"
            )
        )
        .sort(
            "region"
        )
    )

    print(
        consulta_lazy.explain(
            optimized=True
        )
    )

### Qué queremos observar

El plan puede detectar:

- columnas realmente necesarias;
- filtro;
- agregación;
- orden.

Todo esto antes de leer y materializar el resultado final.

# 40. Ejecutar lazy

In [ ]:
if POLARS_DISPONIBLE:
    resultado_polars = (
        consulta_lazy
        .collect()
    )

    print(
        resultado_polars
    )

# 41. Comparación equivalente Pandas

In [ ]:
resultado_pandas_filtrado = (
    pd.read_csv(
        ruta_csv,
        usecols=[
            "region",
            "importe"
        ]
    )
    .loc[
        lambda x:
            x["importe"] > 500
    ]
    .groupby(
        "region",
        as_index=False
    )
    .agg(
        facturacion=(
            "importe",
            "sum"
        ),
        operaciones=(
            "importe",
            "size"
        )
    )
    .sort_values(
        "region"
    )
    .reset_index(
        drop=True
    )
)

resultado_pandas_filtrado

# 42. Validar Pandas vs Polars

In [ ]:
if POLARS_DISPONIBLE:
    resultado_pl_pd = (
        resultado_polars
        .to_pandas()
    )

    print(
        "Facturación:",
        np.allclose(
            resultado_pandas_filtrado[
                "facturacion"
            ],
            resultado_pl_pd[
                "facturacion"
            ]
        )
    )

    print(
        "Operaciones:",
        (
            resultado_pandas_filtrado[
                "operaciones"
            ]
            ==
            resultado_pl_pd[
                "operaciones"
            ]
        ).all()
    )

# 43. Benchmark carga completa vs chunks vs Polars lazy

In [ ]:
def benchmark(
    funcion,
    repeticiones=3
):
    tiempos = []

    for _ in range(
        repeticiones
    ):
        inicio = (
            time.perf_counter()
        )

        funcion()

        tiempos.append(
            time.perf_counter()
            - inicio
        )

    tiempos = np.array(
        tiempos
    )

    return {
        "media_ms":
            tiempos.mean()
            * 1000,

        "std_ms":
            tiempos.std()
            * 1000
    }


def pandas_completo_filtrado():
    return (
        pd.read_csv(
            ruta_csv,
            usecols=[
                "region",
                "importe"
            ]
        )
        .loc[
            lambda x:
                x["importe"] > 500
        ]
        .groupby(
            "region",
            as_index=False
        )
        .agg(
            facturacion=(
                "importe",
                "sum"
            ),
            operaciones=(
                "importe",
                "size"
            )
        )
    )


def pandas_chunks_filtrado():
    acumulado = {}

    for chunk in pd.read_csv(
        ruta_csv,
        usecols=[
            "region",
            "importe"
        ],
        chunksize=100_000
    ):
        chunk = chunk.loc[
            chunk[
                "importe"
            ] > 500
        ]

        parcial = (
            chunk.groupby(
                "region"
            )["importe"]
            .agg(
                ["sum", "count"]
            )
        )

        for region, fila in parcial.iterrows():

            if region not in acumulado:
                acumulado[
                    region
                ] = [
                    0.0,
                    0
                ]

            acumulado[
                region
            ][0] += (
                fila["sum"]
            )

            acumulado[
                region
            ][1] += int(
                fila["count"]
            )

    return acumulado


bench_full = benchmark(
    pandas_completo_filtrado
)

bench_chunks = benchmark(
    pandas_chunks_filtrado
)

if POLARS_DISPONIBLE:
    bench_polars = benchmark(
        lambda:
            consulta_lazy.collect()
    )
else:
    bench_polars = {
        "media_ms":
            np.nan,
        "std_ms":
            np.nan
    }

comparacion_final = pd.DataFrame({
    "metodo": [
        "Pandas completo",
        "Pandas chunks",
        "Polars lazy"
    ],
    "media_ms": [
        bench_full[
            "media_ms"
        ],
        bench_chunks[
            "media_ms"
        ],
        bench_polars[
            "media_ms"
        ]
    ]
})

comparacion_final

# 44. Visualizar benchmark

In [ ]:
fig, ax = plt.subplots(
    figsize=(8, 4)
)

ax.bar(
    comparacion_final[
        "metodo"
    ],
    comparacion_final[
        "media_ms"
    ]
)

ax.set(
    title="Procesamiento de fichero grande",
    ylabel="Tiempo medio (ms)"
)

ax.tick_params(
    axis="x",
    rotation=15
)

plt.show()

### Interpretación

Cada enfoque responde a una necesidad distinta:

- **Pandas completo**: simple cuando el fichero cabe en memoria.
- **Chunks**: control explícito del consumo de memoria.
- **Polars lazy**: delega planificación y optimización en el motor.

La elección debe considerar tiempo, memoria, complejidad y mantenibilidad.

# 45. Escalabilidad con distintos tamaños de chunk

In [ ]:
chunk_sizes_bench = [
    25_000,
    50_000,
    100_000,
    250_000
]

filas = []

for size in chunk_sizes_bench:

    def ejecutar(
        size=size
    ):
        total = 0

        for chunk in pd.read_csv(
            ruta_csv,
            usecols=[
                "importe"
            ],
            chunksize=size
        ):
            total += (
                chunk[
                    "importe"
                ]
                .sum()
            )

        return total

    medicion = medir_memoria(
        ejecutar
    )

    filas.append({
        "chunksize":
            size,

        "tiempo_ms":
            medicion[
                "tiempo_ms"
            ],

        "pico_MB":
            medicion[
                "pico_MB"
            ]
    })

tradeoff_chunks = pd.DataFrame(
    filas
)

tradeoff_chunks

# 46. Gráfico tiempo vs chunk size

In [ ]:
fig, ax = plt.subplots(
    figsize=(7, 4)
)

ax.plot(
    tradeoff_chunks[
        "chunksize"
    ],
    tradeoff_chunks[
        "tiempo_ms"
    ],
    marker="o"
)

ax.set(
    title="Chunk size vs tiempo",
    xlabel="Chunk size",
    ylabel="Tiempo (ms)"
)

plt.show()

# 47. Gráfico memoria vs chunk size

In [ ]:
fig, ax = plt.subplots(
    figsize=(7, 4)
)

ax.plot(
    tradeoff_chunks[
        "chunksize"
    ],
    tradeoff_chunks[
        "pico_MB"
    ],
    marker="o"
)

ax.set(
    title="Chunk size vs pico de memoria",
    xlabel="Chunk size",
    ylabel="Pico MB"
)

plt.show()

### Interpretación

El tamaño de chunk introduce un **trade-off**:

- bloques más grandes: menos overhead, más memoria;
- bloques más pequeños: menos memoria, más iteraciones.

La configuración debe medirse sobre el entorno real.

# 48. Mini-práctica integrada

Objetivo:

Procesar un CSV de gran tamaño para obtener:

- facturación por región;
- operaciones;
- media global;
- máximo;
- número de clientes únicos aproximado o exacto según estrategia.

Implementar:

1. carga completa Pandas;
2. Pandas chunks;
3. Polars lazy.

Después:

- medir tiempo;
- medir memoria;
- validar resultados;
- analizar escalabilidad.

# 49. Checklist de procesamiento incremental

- [ ] ¿Realmente necesito cargar todos los datos?
- [ ] ¿Puedo procesar elemento a elemento?
- [ ] ¿Puedo procesar por bloques?
- [ ] ¿Sé qué estadísticas son combinables?
- [ ] ¿Evito media de medias no ponderada?
- [ ] ¿Selecciono solo columnas necesarias?
- [ ] ¿Filtro temprano?
- [ ] ¿El iterador puede agotarse?
- [ ] ¿Necesito recorrer los datos varias veces?
- [ ] ¿Necesito acceso aleatorio?
- [ ] ¿He validado contra una referencia?
- [ ] ¿He medido memoria?
- [ ] ¿He probado distintos chunk sizes?
- [ ] ¿He considerado Polars lazy?
- [ ] ¿He inspeccionado el plan?

# 50. Ejercicios finales

## Ejercicio 1 — Iterador
Crea una lista, conviértela con `iter()` y demuestra `StopIteration`.

## Ejercicio 2 — Generator expression
Compara una list comprehension y un generator expression.

## Ejercicio 3 — `yield`
Crea un generador que produzca números primos o cuadrados.

## Ejercicio 4 — Memoria
Compara pico de memoria al consumir lista y generador.

## Ejercicio 5 — Fichero línea a línea
Compara `readlines()` con iteración directa sobre un fichero.

## Ejercicio 6 — CSV chunks
Procesa un CSV por bloques y calcula suma total.

## Ejercicio 7 — Media de medias
Construye chunks de tamaños diferentes y demuestra el error.

## Ejercicio 8 — Media incremental
Corrige el ejercicio anterior con suma y conteo.

## Ejercicio 9 — Agregación por grupo
Acumula suma y conteo por región mediante chunks.

## Ejercicio 10 — Validación
Compara el resultado incremental con una carga completa.

## Ejercicio 11 — Column pruning
Lee únicamente las columnas necesarias y mide memoria.

## Ejercicio 12 — Chunk size
Compara al menos cuatro tamaños de chunk.

## Ejercicio 13 — Pipeline de generadores
Construye tres generadores encadenados sin listas intermedias.

## Ejercicio 14 — Polars scan
Implementa una consulta con `scan_csv`, filtro y group_by.

## Ejercicio 15 — Explain
Inspecciona el plan lazy y describe qué optimizaciones observas.

## Ejercicio 16 — Benchmark
Compara Pandas completo, chunks y Polars lazy.

## Ejercicio 17 — Reto final
Genera o utiliza un CSV de al menos un millón de filas. Debes:

1. medir tamaño del fichero;
2. establecer baseline completo;
3. medir memoria;
4. procesar por chunks;
5. evitar media de medias;
6. combinar estadísticas parciales;
7. probar cuatro chunk sizes;
8. seleccionar columnas en lectura;
9. filtrar temprano;
10. implementar Polars lazy;
11. inspeccionar plan;
12. validar equivalencia;
13. comparar tiempos;
14. comparar memoria;
15. justificar el enfoque final.

# 51. Preguntas de reflexión

1. ¿Qué diferencia hay entre iterable e iterador?
2. ¿Qué significa que un iterador se agote?
3. ¿Qué hace `yield`?
4. ¿Por qué un generador puede ahorrar memoria?
5. ¿Un generador siempre es más rápido?
6. ¿Cuándo necesitamos materializar una lista?
7. ¿Por qué `readlines()` puede ser problemático?
8. ¿Qué aporta `chunksize`?
9. ¿Por qué una media de medias puede ser incorrecta?
10. ¿Qué estadísticas pueden combinarse fácilmente entre chunks?
11. ¿Qué trade-off introduce el chunk size?
12. ¿Por qué leer solo algunas columnas puede mejorar el pipeline?
13. ¿Qué diferencia conceptual hay entre chunks manuales y `scan_csv` lazy?
14. ¿Qué aporta el optimizador de Polars?
15. ¿Cómo decidirías entre Pandas completo, chunks y Polars lazy?

# 52. Limpieza de ficheros temporales

In [ ]:
# Ejecutar al final del laboratorio si se desea liberar espacio.

ruta_txt.unlink(
    missing_ok=True
)

ruta_csv.unlink(
    missing_ok=True
)

print(
    "Ficheros temporales eliminados."
)